In [51]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import geopandas as gpd
import contextily as ctx
import rasterio
from tqdm import tqdm

In [52]:
eq_scenario_id = "bssc2014haywardhnhsshaw09mod_m7p32_se"
expansion_scenario = "conservative"

In [53]:
pypsa_data_dir = "/resnick/groups/enceladus/jyzhao/CAPowerResilience/Data/eq_handoff_fullcats"
agg_type = "disagg"
asset_fp = os.path.join(pypsa_data_dir, expansion_scenario, agg_type, "assets_all.csv")
asset_df = pd.read_csv(asset_fp)

In [54]:
line_ind = asset_df[asset_df['bus']=='CATS_1875'].index[0]
site_lon, site_lat = asset_df['lon'][line_ind], asset_df['lat'][line_ind]
print(f"Site longitude: {site_lon}, Site latitude: {site_lat}")

Site longitude: -117.0698252, Site latitude: 33.68098774


In [55]:
eq_scenario_dir = f"/resnick/groups/enceladus/jyzhao/CAPowerResilience/Data/BSSC2014_scenarios/{eq_scenario_id}"
pga_raster_fp = os.path.join(eq_scenario_dir, "raster", "pga.fit")
# 1. Open the file
with rasterio.open(pga_raster_fp,
    crs="EPSG:4326") as src:
    # Query metadata
    print("Metadata Summary:")
    print(f"Dimensions: {src.width}x{src.height}")
    print(f"Number of bands: {src.count}")
    print(f"Coordinate Reference System (CRS): {src.crs}")
    
    # 2. Query and read pixel data (e.g., Band 1)
    # raster_data = src.read(1)
    row, col = src.index(site_lon, site_lat)
    print(f"Pixel coordinates for the site: row={row}, col={col}")
    # 3. Read the pixel value for the site
    if 0 <= row < src.height and 0 <= col < src.width:
        site_pga = src.read(1)[row, col]
        print(f"PGA value for the site: {site_pga} % g")
    else:
        site_pga = 0.0
        print("Site is outside the raster bounds.")

Metadata Summary:
Dimensions: 642x540
Number of bands: 1
Coordinate Reference System (CRS): None
Pixel coordinates for the site: row=761, col=917
Site is outside the raster bounds.


In [56]:
asset_df['site_pga'] = 0.0

In [57]:
with rasterio.open(pga_raster_fp,
    crs="EPSG:4326") as src:
    # Query metadata
    print("Metadata Summary:")
    print(f"Dimensions: {src.width}x{src.height}")
    print(f"Number of bands: {src.count}")
    print(f"Coordinate Reference System (CRS): {src.crs}")
    
    # 2. Query and read pixel data (e.g., Band 1)
    for site_id in tqdm(asset_df.index, total=len(asset_df)):
        site_lon, site_lat = asset_df['lon'][site_id], asset_df['lat'][site_id]
        row, col = src.index(site_lon, site_lat)
        # 3. Read the pixel value for the site
        if 0 <= row < src.height and 0 <= col < src.width:
            site_pga = src.read(1)[row, col]
            asset_df.loc[site_id, 'site_pga'] = site_pga
            # print(f"PGA value for the site: {site_pga}")
        else:
            site_pga = 0.0
            asset_df.loc[site_id, 'site_pga'] = site_pga
            # print("Site is outside the raster bounds.")

Metadata Summary:
Dimensions: 642x540
Number of bands: 1
Coordinate Reference System (CRS): None


  0%|          | 0/3031 [00:00<?, ?it/s]

100%|██████████| 3031/3031 [00:04<00:00, 688.07it/s] 


In [58]:
asset_df['site_pga'] = asset_df['site_pga'] /100 # convert from %g to g
asset_df['asset_id'] = range(len(asset_df))
print(asset_df['site_pga'].describe())

count    3031.000000
mean        0.034488
std         0.085107
min         0.000000
25%         0.000000
50%         0.000000
75%         0.020231
max         0.625595
Name: site_pga, dtype: float64


In [59]:
output_dir = os.path.join(eq_scenario_dir, expansion_scenario)
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
output_fp = os.path.join(output_dir, "assets_all_with_pga.csv")
asset_df.to_csv(output_fp, index=False)

In [60]:
output_dir

'/resnick/groups/enceladus/jyzhao/CAPowerResilience/Data/BSSC2014_scenarios/bssc2014haywardhnhsshaw09mod_m7p32_se/conservative'